In [130]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
# ============================================================
# 第一层：参数区
# ============================================================

START_DATE = "2026-04-01"
END_DATE = "2026-04-30"
NOTICE_TYPE = "all"  # "transfer", "result", or "all"

BASE_OUTPUT_DIR = Path(r"D:\银河\战略发展部\银登中心数据")
RUN_FOLDER_NAME = "2026年4月_银登中心公告下载"
OUTPUT_ROOT = BASE_OUTPUT_DIR / RUN_FOLDER_NAME

REQUEST_TIMEOUT = 20
SLEEP_SECONDS = 0.8
MAX_PAGES_PER_SECTION = 1000
PAGE_SIZE = 10



NOTICE_CONFIG = {
    "transfer": {
        "notice_type": "转让公告",
        "list_url": "https://www.yindeng.com.cn/xxpl/xxpl_bldkzr/bldkzr_zrgg/",
        "folder_name": "转让公告",
        "detail_path_keyword": "/xxpl/xxpl_bldkzr/bldkzr_zrgg/"
    },
    "result": {
        "notice_type": "转让结果公告",
        "list_url": "https://www.yindeng.com.cn/xxpl/xxpl_bldkzr/bldkzr_zrjggg/",
        "folder_name": "转让结果公告",
        "detail_path_keyword": "/xxpl/xxpl_bldkzr/bldkzr_zrjggg/"
    }
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    )
}

In [132]:
# ============================================================
# Step 2. 通用函数
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def sanitize_filename(filename: str) -> str:
    filename = re.sub(r'[\\/:*?"<>|]+', "_", filename)
    filename = re.sub(r"\s+", " ", filename).strip()
    return filename


def fetch_html(url: str) -> str:
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    response.encoding = response.apparent_encoding
    return response.text


def download_file(file_url: str, save_path: Path):
    response = requests.get(file_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(response.content)


def to_date(date_str: str):
    return pd.to_datetime(date_str).date()


def in_date_range(target_date, start_date, end_date):
    return start_date <= target_date <= end_date

In [133]:
# ============================================================
# Step 3. 栏目抓取器
# ============================================================

def extract_total_count(list_html: str):
    """
    提取栏目总条数，例如“共4810条”。
    """
    soup = BeautifulSoup(list_html, "html.parser")
    full_text = soup.get_text(" ", strip=True)

    match = re.search(r"共\s*(\d+)\s*条", full_text)
    if match:
        return int(match.group(1))
    return None


def build_page_url(base_list_url: str, page_no: int, total_pages: int):
    """
    构造栏目分页 URL。

    规则：
    - 第1页：栏目首页
    - 第2页：index_{total_pages - 1}.html
    - 第3页：index_{total_pages - 2}.html
    - ...
    """
    base_list_url = base_list_url.rstrip("/") + "/"

    if page_no == 1:
        return base_list_url

    suffix = total_pages - page_no + 1

    if suffix <= 0:
        return urljoin(base_list_url, "index.html")

    return urljoin(base_list_url, f"index_{suffix}.html")


def parse_notice_links_from_list_page(list_html: str, base_url: str, detail_path_keyword: str):
    """
    从栏目分页页提取公告详情页链接。
    """
    soup = BeautifulSoup(list_html, "html.parser")
    notice_links = []

    detail_pattern = re.compile(
        re.escape(detail_path_keyword) + r"\d{6}/t\d{8}_\d+\.html"
    )

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href", "").strip()
        title = a_tag.get_text(strip=True)

        if not href or not title:
            continue

        if not detail_pattern.search(href):
            continue

        detail_url = urljoin(base_url, href)

        notice_links.append({
            "title": title,
            "detail_url": detail_url
        })

    unique_links = []
    seen = set()

    for item in notice_links:
        key = (item["title"], item["detail_url"])
        if key not in seen:
            seen.add(key)
            unique_links.append(item)

    return unique_links


In [134]:
# ============================================================
# Step 4. 详情页过滤器
# ============================================================

def extract_detail_info_from_detail_page(detail_html: str, detail_url: str):
    """
    从详情页提取发布日期与 PDF 链接。
    """
    soup = BeautifulSoup(detail_html, "html.parser")
    full_text = soup.get_text(" ", strip=True)

    date_match = re.search(r"\d{4}-\d{2}-\d{2}", full_text)
    publish_date = date_match.group(0) if date_match else ""

    pdf_url = None
    pdf_name = None

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href", "").strip()
        text = a_tag.get_text(strip=True)

        if ".pdf" in href.lower() or text.lower().endswith(".pdf"):
            pdf_url = urljoin(detail_url, href)
            pdf_name = text if text else "附件.pdf"
            break

    return {
        "publish_date": publish_date,
        "pdf_url": pdf_url,
        "pdf_name": pdf_name
    }


def build_pdf_save_path(output_root: Path, folder_name: str, publish_date: str, title: str):
    """
    生成 PDF 保存路径：
    根目录 / 公告类型 / YYYY-MM / 发布日期_标题.pdf
    """
    if publish_date:
        month_folder = pd.to_datetime(publish_date).strftime("%Y-%m")
    else:
        month_folder = "unknown_date"

    save_dir = output_root / folder_name / month_folder
    ensure_dir(save_dir)

    safe_title = sanitize_filename(title) if title else "未命名公告"
    file_name = f"{publish_date}_{safe_title}.pdf".strip("_")

    return save_dir / file_name

In [135]:
# ============================================================
# Step 5. 单个栏目抓取
# ============================================================

def crawl_one_section(section_key, start_date, end_date, output_root):
    config = NOTICE_CONFIG[section_key]

    notice_type = config["notice_type"]
    base_list_url = config["list_url"]
    folder_name = config["folder_name"]
    detail_path_keyword = config["detail_path_keyword"]

    print(f"\n{'=' * 70}")
    print(f"开始抓取栏目：{notice_type}")
    print(f"栏目首页：{base_list_url}")
    print(f"日期范围：{start_date} ~ {end_date}")
    print(f"{'=' * 70}")

    log_rows = []

    try:
        first_page_html = fetch_html(base_list_url)
    except Exception as e:
        log_rows.append({
            "notice_type": notice_type,
            "page_no": None,
            "page_url": base_list_url,
            "title": None,
            "detail_url": None,
            "publish_date": None,
            "pdf_url": None,
            "pdf_local_path": None,
            "status": f"first_page_failed: {e}"
        })
        return log_rows

    total_count = extract_total_count(first_page_html)

    if total_count is None:
        print("未识别到总条数，按 MAX_PAGES_PER_SECTION 上限抓取。")
        total_pages = MAX_PAGES_PER_SECTION
    else:
        total_pages = min((total_count + PAGE_SIZE - 1) // PAGE_SIZE, MAX_PAGES_PER_SECTION)
        print(f"识别到总条数：{total_count}，预计分页数：{total_pages}")

    for page_no in range(1, total_pages + 1):
        page_url = build_page_url(base_list_url, page_no, total_pages)
        print(f"\n正在处理第 {page_no} 页：{page_url}")

        try:
            if page_no == 1:
                list_html = first_page_html
            else:
                time.sleep(SLEEP_SECONDS)
                list_html = fetch_html(page_url)
        except Exception as e:
            print(f"分页请求失败：{e}")
            log_rows.append({
                "notice_type": notice_type,
                "page_no": page_no,
                "page_url": page_url,
                "title": None,
                "detail_url": None,
                "publish_date": None,
                "pdf_url": None,
                "pdf_local_path": None,
                "status": f"page_failed: {e}"
            })
            continue

        notice_links = parse_notice_links_from_list_page(
            list_html=list_html,
            base_url=page_url,
            detail_path_keyword=detail_path_keyword
        )

        print(f"本页解析到 {len(notice_links)} 条详情页链接。")

        if not notice_links:
            print("本页未解析到详情页链接，继续下一页。")
            continue

        page_oldest_date = None
        page_has_in_range = False

        for idx, notice in enumerate(notice_links, start=1):
            title = notice["title"]
            detail_url = notice["detail_url"]

            print(f"  [{idx}/{len(notice_links)}] 处理详情页：{title}")

            try:
                time.sleep(SLEEP_SECONDS)
                detail_html = fetch_html(detail_url)
                detail_info = extract_detail_info_from_detail_page(detail_html, detail_url)

                publish_date_str = detail_info["publish_date"]
                pdf_url = detail_info["pdf_url"]

                if not publish_date_str:
                    log_rows.append({
                        "notice_type": notice_type,
                        "page_no": page_no,
                        "page_url": page_url,
                        "title": title,
                        "detail_url": detail_url,
                        "publish_date": None,
                        "pdf_url": pdf_url,
                        "pdf_local_path": None,
                        "status": "publish_date_not_found"
                    })
                    continue

                publish_date = to_date(publish_date_str)

                if page_oldest_date is None or publish_date < page_oldest_date:
                    page_oldest_date = publish_date

                if not in_date_range(publish_date, start_date, end_date):
                    log_rows.append({
                        "notice_type": notice_type,
                        "page_no": page_no,
                        "page_url": page_url,
                        "title": title,
                        "detail_url": detail_url,
                        "publish_date": publish_date_str,
                        "pdf_url": pdf_url,
                        "pdf_local_path": None,
                        "status": "out_of_range"
                    })
                    continue

                page_has_in_range = True

                if not pdf_url:
                    log_rows.append({
                        "notice_type": notice_type,
                        "page_no": page_no,
                        "page_url": page_url,
                        "title": title,
                        "detail_url": detail_url,
                        "publish_date": publish_date_str,
                        "pdf_url": None,
                        "pdf_local_path": None,
                        "status": "pdf_not_found"
                    })
                    continue

                save_path = build_pdf_save_path(
                    output_root=output_root,
                    folder_name=folder_name,
                    publish_date=publish_date_str,
                    title=title
                )

                if save_path.exists():
                    status = "already_exists"
                else:
                    download_file(pdf_url, save_path)
                    status = "downloaded"

                log_rows.append({
                    "notice_type": notice_type,
                    "page_no": page_no,
                    "page_url": page_url,
                    "title": title,
                    "detail_url": detail_url,
                    "publish_date": publish_date_str,
                    "pdf_url": pdf_url,
                    "pdf_local_path": str(save_path),
                    "status": status
                })

            except Exception as e:
                log_rows.append({
                    "notice_type": notice_type,
                    "page_no": page_no,
                    "page_url": page_url,
                    "title": title,
                    "detail_url": detail_url,
                    "publish_date": None,
                    "pdf_url": None,
                    "pdf_local_path": None,
                    "status": f"detail_failed: {e}"
                })

        if page_oldest_date is not None:
            print(f"本页最早日期：{page_oldest_date}")

            if page_oldest_date < start_date and not page_has_in_range:
                print("本页已经整体早于目标时间范围，停止该栏目翻页。")
                break

    return log_rows

In [136]:
# ============================================================
# Step 6. 日志输出
# ============================================================

def save_log(log_rows, output_root: Path):
    log_dir = output_root / "logs"
    ensure_dir(log_dir)

    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"yindeng_download_log_{ts}.csv"

    df_log = pd.DataFrame(log_rows)
    df_log.to_csv(log_path, index=False, encoding="utf-8-sig")

    print(f"\n日志已保存：{log_path}")
    return log_path


In [137]:
# ============================================================
# Step 7. 主程序
# ============================================================

def main():
    start_date = to_date(START_DATE)
    end_date = to_date(END_DATE)

    if start_date > end_date:
        raise ValueError("START_DATE 不能晚于 END_DATE。")

    ensure_dir(OUTPUT_ROOT)

    if NOTICE_TYPE == "transfer":
        section_keys = ["transfer"]
    elif NOTICE_TYPE == "result":
        section_keys = ["result"]
    elif NOTICE_TYPE == "all":
        section_keys = ["transfer", "result"]
    else:
        raise ValueError("NOTICE_TYPE 只能是 'transfer'、'result' 或 'all'。")

    all_log_rows = []

    for section_key in section_keys:
        section_log_rows = crawl_one_section(
            section_key=section_key,
            start_date=start_date,
            end_date=end_date,
            output_root=OUTPUT_ROOT
        )
        all_log_rows.extend(section_log_rows)

    log_path = save_log(all_log_rows, OUTPUT_ROOT)

    df_log = pd.DataFrame(all_log_rows)

    print("\n" + "=" * 70)
    print("抓取完成，结果汇总：")
    print("=" * 70)

    if not df_log.empty:
        print(df_log["status"].value_counts(dropna=False))
    else:
        print("无日志记录。")

    print(f"\n输出目录：{OUTPUT_ROOT}")
    print(f"日志文件：{log_path}")


if __name__ == "__main__":
    main()


开始抓取栏目：转让公告
栏目首页：https://www.yindeng.com.cn/xxpl/xxpl_bldkzr/bldkzr_zrgg/
日期范围：2026-04-01 ~ 2026-04-30
识别到总条数：4810，预计分页数：481

正在处理第 1 页：https://www.yindeng.com.cn/xxpl/xxpl_bldkzr/bldkzr_zrgg/
本页解析到 10 条详情页链接。
  [1/10] 处理详情页：中原银行股份有限公司关于2026年第1期个人不良贷款（个人消费及经营性贷款）转让项目转让公告
  [2/10] 处理详情页：甘肃农村商业银行股份有限公司关于2026年庆阳市玉霖食品有限责任公司不良贷款转让项目转让公告
  [3/10] 处理详情页：平安银行股份有限公司关于2026年第15期个人不良贷款（信用卡透支）转让项目转让公告
  [4/10] 处理详情页：中原银行股份有限公司郑州分行关于2026年第2期个人不良贷款（个人消费及经营性贷款）转让项目转让公告
  [5/10] 处理详情页：中银消费金融有限公司关于2026年第10期个人不良贷款（个人消费贷款）转让项目转让公告
  [6/10] 处理详情页：宁波银行股份有限公司关于2026年第1期个人不良贷款（个人经营性贷款）转让项目转让公告
  [7/10] 处理详情页：中国银行股份有限公司苏州分行关于2026年第3期个人不良贷款（信用卡透支）转让项目转让公告
  [8/10] 处理详情页：中国农业银行股份有限公司广东省分行关于2026年第1期个人不良贷款（个人经营性贷款）转让项目转让公告
  [9/10] 处理详情页：中国建设银行股份有限公司福建省分行关于2026年第3期个人不良贷款（个人消费及经营性贷款）转让项目转让公告
  [10/10] 处理详情页：交通银行股份有限公司山东省分行关于2026年第1期个人不良贷款（个人消费贷款）转让项目转让公告
本页最早日期：2026-04-14

正在处理第 2 页：https://www.yindeng.com.cn/xxpl/xxpl_bldkzr/bldkzr_zrgg/index_480.html
本页解析到 10 条详情页链接。
  [1/10] 处理详情页：平安银行股份有限公司关于2026年第14期个人不良贷款（信用卡